## Joins at Scale: Broadcasts, Hints and Skew Tutoriail

#### Data Generation

In [0]:
from pyspark.sql import functions as F

# Small enough for free serverless, big enough to show joins/shuffles
N_ORDERS     = 30000
N_CUSTOMERS  = 6000
N_CATEGORIES = 30

spark.conf.set("spark.sql.shuffle.partitions", 8)  # keep plans readable for demos

In [0]:
# dim_regions: tiny ⇒ great to broadcast
dim_regions = spark.createDataFrame(
    [(1,"North America"), (2,"Europe"), (3,"Asia"), (4,"South America"), (5,"Africa"), (6,"Oceania")],
    ["region_id","region_name"]
)

# dim_categories: small/medium; OK for joins
dim_categories = (
    spark.range(1, N_CATEGORIES + 1)
         .select(
             F.col("id").cast("int").alias("category_id"),
             F.concat(F.lit("Cat-"), F.col("id")).alias("category_name")
         )
)

# dim_customers: medium, with SKEWED region assignment (more customers in region_id=1)
u = F.rand(7)
dim_customers = (
    spark.range(1, N_CUSTOMERS + 1)
         .select(
             F.col("id").cast("int").alias("customer_id"),
             F.when(u < 0.55, 1)  # ~55% in region 1
              .when(u < 0.70, 2)  # ~15% region 2
              .when(u < 0.82, 3)  # ~12% region 3
              .when(u < 0.92, 4)  # ~10% region 4
              .when(u < 0.98, 5)  # ~6% region 5
              .otherwise(6)       # ~2% region 6
              .alias("region_id"),
             F.when(F.rand(8) < 0.10, "Platinum")
              .when(F.rand(8) < 0.40, "Gold")
              .otherwise("Silver").alias("loyalty")
         )
)

In [0]:
# orders: references customer_id and category_id; no region_id here (you'll join via customers)
orders = (
    spark.range(N_ORDERS)
         .select(
             (F.col("id") + 1).cast("long").alias("order_id"),
             (F.rand(42) * N_CUSTOMERS + 1).cast("int").alias("customer_id"),
             (F.rand(43) * N_CATEGORIES + 1).cast("int").alias("category_id"),
             F.round(F.rand(44) * 500 + 20, 2).alias("amount"),
             F.to_date(F.expr("date_add('2015-01-01', CAST(rand(45)*3650 AS INT))")).alias("order_date")
         )
)


In [0]:
orders.createOrReplaceTempView("fact_orders")
dim_customers.createOrReplaceTempView("dim_customers")
dim_regions.createOrReplaceTempView("dim_regions")
dim_categories.createOrReplaceTempView("dim_categories")

# quick peeks
display(orders.limit(5))
display(dim_customers.groupBy("region_id").count().orderBy(F.desc("count")))  # shows skew


order_id,customer_id,category_id,amount,order_date
1,515,17,401.41,2023-03-15
2,1863,20,60.11,2020-03-02
3,376,21,270.44,2022-10-10
4,1839,9,144.92,2016-06-27
5,1,27,258.01,2017-05-19


region_id,count
1,3355
2,1879
3,648
4,111
5,7


In [0]:
orders.count()

30000

#### Load Data in Dataframes

In [0]:
from pyspark.sql import functions as F

# keep plans readable
spark.conf.set("spark.sql.shuffle.partitions", 8)

# pull the DataFrames from the temp views / tables you created earlier
orders        = spark.table("fact_orders")
dim_customers = spark.table("dim_customers")
dim_regions   = spark.table("dim_regions")
dim_categories= spark.table("dim_categories")

display(orders.limit(5))
display(dim_customers.limit(5))


order_id,customer_id,category_id,amount,order_date
1,515,17,401.41,2023-03-15
2,1863,20,60.11,2020-03-02
3,376,21,270.44,2022-10-10
4,1839,9,144.92,2016-06-27
5,1,27,258.01,2017-05-19


customer_id,region_id,loyalty
1,5,Gold
2,1,Silver
3,2,Gold
4,2,Gold
5,3,Gold


#### This shows data skew

In [0]:
display(dim_customers.groupBy("region_id").count().orderBy(F.desc("count")))  # shows skew on region 1

region_id,count
1,3355
2,1879
3,648
4,111
5,7


#### Broadcast Hash Join

In [0]:
j_auto = orders.join(dim_customers, on="customer_id", how="inner")
j_auto.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- == Initial Plan ==
   ColumnarToRow (12)
   +- PhotonResultStage (11)
      +- PhotonProject (10)
         +- PhotonBroadcastHashJoin Inner (9)
            :- PhotonFilter (3)
            :  +- PhotonProject (2)
            :     +- PhotonRange (1)
            +- PhotonShuffleExchangeSource (8)
               +- PhotonShuffleMapStage (7)
                  +- PhotonShuffleExchangeSink (6)
                     +- PhotonProject (5)
                        +- PhotonRange (4)


(1) PhotonRange
Output [1]: [id#10999L]
Arguments: Range (0, 30000, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#10999L]
Arguments: [(id#10999L + 1) AS order_id#11000L, cast(((rand(42) * 6000.0) + 1.0) as int) AS customer_id#11001, cast(((rand(43) * 30.0) + 1.0) as int) AS category_id#11002, round(((rand(44) * 500.0) + 20.0), 2) AS amount#11003, date_add(2015-01-01, cast((rand(45) * 3650.0) as int)) AS order_date#11004]

(3) PhotonFilter
Input [5]: [order_id#11000L

In [0]:
from pyspark.sql.functions import broadcast
j_bcast = orders.join(broadcast(dim_customers), on="customer_id", how="inner")
j_bcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- == Initial Plan ==
   ColumnarToRow (12)
   +- PhotonResultStage (11)
      +- PhotonProject (10)
         +- PhotonBroadcastHashJoin Inner (9)
            :- PhotonFilter (3)
            :  +- PhotonProject (2)
            :     +- PhotonRange (1)
            +- PhotonShuffleExchangeSource (8)
               +- PhotonShuffleMapStage (7)
                  +- PhotonShuffleExchangeSink (6)
                     +- PhotonProject (5)
                        +- PhotonRange (4)


(1) PhotonRange
Output [1]: [id#10999L]
Arguments: Range (0, 30000, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#10999L]
Arguments: [(id#10999L + 1) AS order_id#11000L, cast(((rand(42) * 6000.0) + 1.0) as int) AS customer_id#11001, cast(((rand(43) * 30.0) + 1.0) as int) AS category_id#11002, round(((rand(44) * 500.0) + 20.0), 2) AS amount#11003, date_add(2015-01-01, cast((rand(45) * 3650.0) as int)) AS order_date#11004]

(3) PhotonFilter
Input [5]: [order_id#11000L

#### Hints to force Shuffle Hash

In [0]:
j_shuffle = orders.join(dim_customers.hint("shuffle_hash"),
                       on="customer_id", how="inner")

j_shuffle.explain("formatted")
display(j_shuffle.limit(5))

== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   ColumnarToRow (15)
   +- PhotonResultStage (14)
      +- PhotonProject (13)
         +- PhotonShuffledHashJoin Inner (12)
            :- PhotonShuffleExchangeSource (6)
            :  +- PhotonShuffleMapStage (5)
            :     +- PhotonShuffleExchangeSink (4)
            :        +- PhotonFilter (3)
            :           +- PhotonProject (2)
            :              +- PhotonRange (1)
            +- PhotonShuffleExchangeSource (11)
               +- PhotonShuffleMapStage (10)
                  +- PhotonShuffleExchangeSink (9)
                     +- PhotonProject (8)
                        +- PhotonRange (7)


(1) PhotonRange
Output [1]: [id#10999L]
Arguments: Range (0, 30000, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#10999L]
Arguments: [(id#10999L + 1) AS order_id#11000L, cast(((rand(42) * 6000.0) + 1.0) as int) AS customer_id#11001, cast(((rand(43) * 30.0) + 1.0) as int) AS category_id#11002, round

customer_id,order_id,category_id,amount,order_date,region_id,loyalty
1839,4,9,144.92,2016-06-27,1,Silver
5177,9,27,516.95,2020-08-06,2,Platinum
1266,11,11,77.01,2017-06-28,1,Gold
5930,25,20,470.09,2017-09-22,1,Silver
5796,26,26,242.23,2015-02-09,1,Platinum
